** Prefect conversion to git storage **
-- use for converting all files or individual files to git storage

In [1]:
# set vars
folderPath = r"C:\Source\Repos\EnterpriseCloud\Prefect"
tempDir = folderPath + "\PrefectTemp"


In [2]:
#imports

import os
import shutil
import fnmatch

In [3]:
# helper functions - run this first

def getFileList(path):
    # get file list for path
    filesList = []
    fileNames = []
    for x in os.listdir(path):
        z = x.upper()
        if z.endswith('.PY'):
            fileNames.append(x)
            filesList.append(path + "\\" + x)
    return filesList, fileNames

def getFile(path):
    f = open(path,'r')
    filein = f.readlines()
    f.close()
    return filein

def writeFile(path,fileOut):
    f = open(path,'w')
    f.write(fileOut)
    f.close()

In [4]:
#validate imports, add os and prefect.storage is not already

fileList, fileName = getFileList(folderPath)
fileNameIdx = 0

for f in fileList:
    filein = getFile(f)
    # print(f,filein)
    addOSImport = True
    addStorageImport = True
    fileidx = len(filein)

    ios = fnmatch.filter(filein,'import os*')
    if len(ios) == 0:
        filein.insert(0,'import os\n')

    stor = fnmatch.filter(filein,'from prefect import *storage\n')
    if len(stor) == 0:
        filein.insert(1,'from prefect import storage\n')

    git = fnmatch.filter(filein,'flow.storage = storage.Git*')
    if len(git) == 0:
            filein.insert(fileidx+1,'flow.storage = storage.Git(flow_path="prefect/' + fileName[fileNameIdx] + '",\n')
            filein.insert(fileidx+2,'                         repo="EnterpriseCloud",\n')
            filein.insert(fileidx+3,'                         git_clone_url_secret_name="azure_devops_git_url",\n')
            filein.insert(fileidx+4,'                         branch_name="<<BRANCH>>")\n')
    
    # remove existing lines we don't want
    fileOut = []
    for l in filein:
        # remove flow registration
        if "flow.register(" in l:
            continue
        fileOut.append(l)

    #string out the file line list
    out = ""
    for l in fileOut: out += l

    fileNameIdx += 1
    writeFile(f,out)


In [4]:
# copy out files prefect files to temp folder

if not os.path.exists(tempDir):
    os.mkdir(tempDir)

fileList, fileName = getFileList(folderPath)
idx = 0
for f in fileList:
    newfile = tempDir + "\\" + fileName[idx]
    if not os.path.exists(newfile):
        shutil.copy(f,newfile)
        idx +=1


In [5]:
# run token replacer
run =  f'python ..\\Python\\token_replacer.py \
    --environment "SIT" \
    --build_directory ..\\Prefect\PrefectTemp \
    --config_file "..\\Configuration\\prefect.json" \
    --file_name_pattern "\\*.py"' 
print(run)
 
!{run}


python ..\Python\token_replacer.py     --environment "SIT"     --build_directory ..\Prefect\PrefectTemp     --config_file "..\Configuration\prefect.json"     --file_name_pattern "\*.py"
build_directory: ..\Prefect\PrefectTemp
environment: SIT
config_file:..\Configuration\prefect.json
file_names:
['..\\Prefect\\PrefectTemp\\DBT.Common-PUB.CommandExecution.py', '..\\Prefect\\PrefectTemp\\DBT.Driver.CRM.py', '..\\Prefect\\PrefectTemp\\DBT.Driver.CRMSales.py', '..\\Prefect\\PrefectTemp\\DBT.Driver.GMX.py', '..\\Prefect\\PrefectTemp\\DBT.Driver.Prefect.py', '..\\Prefect\\PrefectTemp\\DBT.Driver.py', '..\\Prefect\\PrefectTemp\\DBT.Driver.SAPAPI.py', '..\\Prefect\\PrefectTemp\\DBT.Driver.SIRFIS.py', '..\\Prefect\\PrefectTemp\\DBT.Driver.TELCustomer.py', '..\\Prefect\\PrefectTemp\\DBT.Driver.Testing.py', '..\\Prefect\\PrefectTemp\\DBT.Driver.Validator.py', '..\\Prefect\\PrefectTemp\\DBT.LZ-ODS.CommandExecution.py', '..\\Prefect\\PrefectTemp\\DBT.ODS-Common.CommandExecution.py', '..\\Prefect\\P

In [75]:
# run registration from prefect cli to loop through all files in prefectTEmp folder.





TypeError: __init__() got an unexpected keyword argument 'folderPath'